# 🧠 Módulo 02: Deep Learning "From Scratch" & El Grafo Computacional
## Capítulo 2: Multilayer Perceptron (MLP) y Backpropagation Vectorial Paso a Paso

> *"En Micrograd aprendimos cómo funciona la diferenciación automática escalar. Pero el hardware moderno no computa flotante a flotante: procesa lotes enteros de tensores mediante multiplicación matricial masiva (GEMM). Hoy damos el salto definitivo: derivar e implementar el Backpropagation vectorial puro para un MLP completo."*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mcarbonell/algo-to-ai/blob/main/notebooks/02_dl_from_scratch/02_vectorized_mlp_backprop.ipynb)

---

### ⚙️ Inicialización del Entorno
Cargamos librerías estándar y fijamos semillas para asegurar reproducibilidad determinista.

In [ ]:
# !pip install -q numpy matplotlib torch
from typing import Tuple, List, Dict
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
print("✅ Entorno listo para Backpropagation Vectorial y MLP from scratch")

---

## 1. 📜 Contexto Histórico y Proceso de Descubrimiento

### El Límite de los Escalares y la Llegada de las GPUs
En los años 80 y 90, la mayoría de los investigadores programaban redes neuronales mediante bucles anidados en C o Fortran: iteraban neurona por neurona y muestra por muestra. Este enfoque presentaba un muro insalvable de rendimiento:
* **Pésimo uso de memoria:** Miles de lecturas dispersas en la memoria RAM.
* **Desperdicio de paralelismo:** Las CPUs modernas y, especialmente, las GPUs (Unidades de Procesamiento Gráfico) son arquitecturas SIMD/SIMT diseñadas para realizar la misma operación sobre millones de datos simultáneamente.

El gran salto del Deep Learning ocurrió cuando nos dimos cuenta de que una capa densa de neuronas no es más que una **única multiplicación de matrices (GEMM - General Matrix Multiply)**:
$$Z = X W + b$$

Al agrupar $B$ ejemplos en una matriz de entrada $X \in \mathbb{R}^{B \times d_{in}}$, procesamos un lote entero (*batch*) en un único ciclo de cómputo altamente optimizado por librerías BLAS (como cuBLAS o Intel MKL).

### La Gran Crisis del Gradiente Desvaneciente (*Vanishing Gradient*)
Durante más de 20 años, la función de activación obligatoria en todas las redes fue la **Sigmoide**:
$$\sigma(z) = \frac{1}{1 + e^{-z}} \quad \implies \quad \sigma'(z) = \sigma(z)(1 - \sigma(z))$$

Observa el valor máximo de su derivada: ocurre en $z = 0$, donde $\sigma'(0) = 0.5 \cdot 0.5 = \mathbf{0.25}$. Conforme $|z|$ crece, la derivada cae exponencialmente a $0$.

Cuando encadenamos $L$ capas densas con sigmoide, la regla de la cadena multiplica sucesivamente las derivadas locales:
$$\frac{\partial \mathcal{L}}{\partial W_1} \propto \prod_{l=1}^L \sigma'(z_l) \le (0.25)^L$$

En una red de solo 5 capas, $(0.25)^5 \approx \mathbf{0.00097}$. El gradiente que llegaba a las primeras capas se desvanecía por completo: **las primeras capas nunca aprendían y las redes profundas se volvían imposibles de entrenar**.

### El Momento Eureka: ReLU (Glorot, Bordes & Bengio, 2011)
En 2011, Xavier Glorot, Antoine Bordes y Yoshua Bengio publicaron un artículo que transformó la disciplina: *"Deep Sparse Rectifier Neural Networks"*.

Demostraron que la función más simple imaginable —la **Unidad Lineal Rectificada (ReLU)**— resolvía de un plumazo el gradiente desvaneciente:
$$\text{ReLU}(z) = \max(0, z) \quad \implies \quad \text{ReLU}'(z) = \begin{cases} 1 & \text{si } z > 0 \\ 0 & \text{si } z \le 0 \end{cases}$$

Para cualquier neurona activa ($z > 0$), la derivada es **exactamente 1.0**: ¡el gradiente fluye hacia atrás a través de cientos de capas sin amortiguación alguna!

---

## 2. 🧠 Intuición Geométrica y Mecánica (Mentalidad de Algoritmista)

### La Anatomía Tensorial de una Capa Densa
Definamos con precisión matemática y dimensional cada variable:
* **Lote de Entrada ($X$):** Matriz de tamaño $(B, d_{in})$, donde $B$ es el tamaño del batch y $d_{in}$ el número de características de entrada.
* **Matriz de Pesos ($W$):** Matriz de tamaño $(d_{in}, d_{out})$.
* **Vector de Sesgo ($b$):** Vector de tamaño $(1, d_{out})$.
* **Activación Lineal ($Z$):** $Z = X W + b \in \mathbb{R}^{B \times d_{out}}$.
* **Activación No Lineal ($A$):** $A = \text{ReLU}(Z) \in \mathbb{R}^{B \times d_{out}}$.

### La Derivación del Backward Matricial (El "Shape Matching Trick")
Supongamos que conocemos el gradiente que baja desde capas superiores respecto a la salida de esta capa: $\frac{\partial \mathcal{L}}{\partial Z} = \delta \in \mathbb{R}^{B \times d_{out}}$.

¿Cómo calculamos los gradientes respecto a $W$, $X$ y $b$?

#### 1. Gradiente de los Pesos: $\frac{\partial \mathcal{L}}{\partial W}$
* Por definición, $\frac{\partial \mathcal{L}}{\partial W}$ debe tener exactamente las mismas dimensiones que $W$: $(d_{in}, d_{out})$.
* Tenemos a nuestra disposición $X \in \mathbb{R}^{B \times d_{in}}$ y $\delta \in \mathbb{R}^{B \times d_{out}}$.
* ¿Cuál es la única multiplicación matricial consistente que produce $(d_{in}, d_{out})$ sumando la contribución de todas las $B$ muestras del batch?
$$\mathbf{\frac{\partial \mathcal{L}}{\partial W} = X^T \delta} \quad \in \mathbb{R}^{d_{in} \times d_{out}}$$

#### 2. Gradiente de la Entrada: $\frac{\partial \mathcal{L}}{\partial X}$
* Debe tener las mismas dimensiones que $X$: $(B, d_{in})$.
* Disponemos de $\delta \in \mathbb{R}^{B \times d_{out}}$ y $W \in \mathbb{R}^{d_{in} \times d_{out}}$.
* La única multiplicación dimensionalmente válida y matemáticamente correcta es:
$$\mathbf{\frac{\partial \mathcal{L}}{\partial X} = \delta W^T} \quad \in \mathbb{R}^{B \times d_{in}}$$

#### 3. Gradiente del Sesgo: $\frac{\partial \mathcal{L}}{\partial b}$
* Dado que el sesgo $b$ se expandió por broadcasting a lo largo de las $B$ filas, su gradiente acumulado es simplemente la suma sobre el eje del batch ($axis=0$):
$$\mathbf{\frac{\partial \mathcal{L}}{\partial b} = \sum_{i=1}^B \delta_{i, :}} \quad \in \mathbb{R}^{1 \times d_{out}}$$

> **Regla Mnemotécnica del Algoritmista:** Si dominas estas tres ecuaciones ($X^T \delta$, $\delta W^T$ y $\sum \delta$), dominas el 90% del cálculo de cualquier red neuronal densa moderna.

In [ ]:
# Visualización de la anatomía del flujo de formas (Shapes)
B, d_in, d_out = 4, 3, 2
print("--- Formas en Propagación Hacia Adelante (Forward) ---")
print(f"Entrada X:           ({B}, {d_in})")
print(f"Pesos W:             ({d_in}, {d_out})")
print(f"Sesgo b:             (1, {d_out})")
print(f"Salida Lineal Z:     ({B}, {d_out}) <- X @ W + b\n")

print("--- Formas en Propagación Hacia Atrás (Backward) ---")
print(f"Gradiente entrante δ: ({B}, {d_out})")
print(f"Gradiente dL/dW:      ({d_in}, {d_out}) <- X.T @ δ (Coincide con W!)")
print(f"Gradiente dL/dX:      ({B}, {d_in}) <- δ @ W.T (Coincide con X!)")
print(f"Gradiente dL/db:      (1, {d_out}) <- sum(δ, axis=0) (Coincide con b!)")

---

## 3. 🛠️ Implementación "From Scratch" (Primeros Principios)

Construyamos un sistema modular estilo PyTorch pero usando **únicamente NumPy** y operaciones matriciales puras.

In [ ]:
class LinearLayer:
    """
    Capa lineal densa vectorizada: Z = X @ W + b
    Implementa forward y backward analítico matricial puro.
    """
    def __init__(self, in_features: int, out_features: int):
        # Inicialización de He/Kaiming para ReLU: varianza = 2 / in_features
        scale = np.sqrt(2.0 / in_features)
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros((1, out_features))
        
        # Almacenamiento de gradientes
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        
        # Caché del forward para el backward
        self.X_cache = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        self.X_cache = X
        return X @ self.W + self.b

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        # grad_output es delta: (B, out_features)
        # 1. dL/dW = X^T @ delta
        self.dW = self.X_cache.T @ grad_output
        # 2. dL/db = sum(delta, axis=0)
        self.db = np.sum(grad_output, axis=0, keepdims=True)
        # 3. dL/dX = delta @ W^T
        grad_input = grad_output @ self.W.T
        return grad_input


class ReLULayer:
    """
    Función de activación ReLU vectorizada: A = max(0, Z)
    """
    def __init__(self):
        self.Z_cache = None

    def forward(self, Z: np.ndarray) -> np.ndarray:
        self.Z_cache = Z
        return np.maximum(0.0, Z)

    def backward(self, grad_output: np.ndarray) -> np.ndarray:
        # dL/dZ = grad_output * (Z > 0)
        return grad_output * (self.Z_cache > 0.0)


class SoftmaxCrossEntropy:
    """
    Función de pérdida de Entropía Cruzada Multiclase con Softmax integrado.
    Numéricamente estable mediante el truco de resta del máximo.
    """
    def __init__(self):
        self.probs_cache = None
        self.y_true_cache = None

    def forward(self, logits: np.ndarray, y_true: np.ndarray) -> float:
        self.y_true_cache = y_true
        B = logits.shape[0]
        
        # Estabilidad numérica: restar el máximo por fila para evitar overflow en exp()
        shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
        exp_scores = np.exp(shifted_logits)
        self.probs_cache = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
        
        # Pérdida de entropía cruzada: - 1/B * sum(log(p_correct))
        correct_log_probs = -np.log(self.probs_cache[np.arange(B), y_true] + 1e-15)
        return float(np.mean(correct_log_probs))

    def backward(self) -> np.ndarray:
        """
        Derivación analítica elegante de Softmax + CrossEntropy:
        dL/dZ = (Probs - OneHot(y)) / B
        """
        B = self.probs_cache.shape[0]
        grad = self.probs_cache.copy()
        grad[np.arange(B), self.y_true_cache] -= 1.0
        return grad / B

print("✅ Módulos LinearLayer, ReLULayer y SoftmaxCrossEntropy compilados")

### Construyendo el Multilayer Perceptron Completo (`VectorizedMLP`)

In [ ]:
class VectorizedMLP:
    """
    Red Neuronal Densa Multicapa completa from-scratch.
    Arquitectura: Linear -> ReLU -> Linear -> ReLU -> Linear (Logits)
    """
    def __init__(self, layer_sizes: List[int]):
        self.layers = []
        for i in range(len(layer_sizes) - 1):
            self.layers.append(LinearLayer(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                self.layers.append(ReLULayer())

    def forward(self, X: np.ndarray) -> np.ndarray:
        out = X
        for layer in self.layers:
            out = layer.forward(out)
        return out

    def backward(self, grad_output: np.ndarray):
        grad = grad_output
        for layer in reversed(self.layers):
            grad = layer.backward(grad)

    def step(self, lr: float):
        """Actualización simple por SGD de todas las capas lineales."""
        for layer in self.layers:
            if isinstance(layer, LinearLayer):
                layer.W -= lr * layer.dW
                layer.b -= lr * layer.db

print("✅ Clase VectorizedMLP definida correctamente")

### Demostración: Resolviendo las Dos Espirales Entrelazadas (*Two Spirals Benchmark*)
El problema de las dos espirales entrelazadas es una prueba clásica de fuego en Deep Learning: ninguna combinación lineal ni árbol de decisión superficial puede resolverlo.

In [ ]:
def generar_espirales(n_puntos: int = 150) -> Tuple[np.ndarray, np.ndarray]:
    """Genera el dataset clásico de dos espirales no lineales."""
    theta = np.sqrt(np.random.rand(n_puntos)) * 2 * np.pi
    r_a = 2 * theta + np.pi
    data_a = np.array([np.cos(theta) * r_a, np.sin(theta) * r_a]).T + np.random.randn(n_puntos, 2) * 0.2
    r_b = -2 * theta - np.pi
    data_b = np.array([np.cos(theta) * r_b, np.sin(theta) * r_b]).T + np.random.randn(n_puntos, 2) * 0.2
    X = np.vstack([data_a, data_b])
    # Normalizar entradas a media 0 y desviación 1 para facilitar convergencia
    X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)
    y = np.array([0] * n_puntos + [1] * n_puntos)
    return X, y

X_spiral, y_spiral = generar_espirales(150)

# Construir un MLP: 2 entradas -> 32 ocultas -> 32 ocultas -> 2 salidas
mlp = VectorizedMLP([2, 32, 32, 2])
loss_fn = SoftmaxCrossEntropy()

losses = []
lr = 0.2
epochs = 800

for epoch in range(epochs):
    # 1. Forward
    logits = mlp.forward(X_spiral)
    loss = loss_fn.forward(logits, y_spiral)
    losses.append(loss)
    
    # 2. Backward
    grad_loss = loss_fn.backward()
    mlp.backward(grad_loss)
    
    # 3. Update
    mlp.step(lr)

preds = np.argmax(mlp.forward(X_spiral), axis=1)
acc = np.mean(preds == y_spiral) * 100
print(f"Entrenamiento finalizado. Pérdida inicial: {losses[0]:.4f} -> Final: {losses[-1]:.4f}")
print(f"🎯 Precisión en espirales no lineales: {acc:.1f}%")

# Gráfica de frontera de decisión aprendida
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Frontera de decisión 2D
x_min, x_max = X_spiral[:, 0].min() - 0.5, X_spiral[:, 0].max() + 0.5
y_min, y_max = X_spiral[:, 1].min() - 0.5, X_spiral[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
grid = np.c_[xx.ravel(), yy.ravel()]
Z_grid = np.argmax(mlp.forward(grid), axis=1).reshape(xx.shape)

ax1.contourf(xx, yy, Z_grid, alpha=0.3, cmap='coolwarm')
ax1.scatter(X_spiral[y_spiral == 0, 0], X_spiral[y_spiral == 0, 1], color='blue', label='Clase 0', alpha=0.8)
ax1.scatter(X_spiral[y_spiral == 1, 0], X_spiral[y_spiral == 1, 1], color='red', label='Clase 1', alpha=0.8)
ax1.set_title("Frontera No Lineal Aprendida por el MLP From-Scratch")
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.4)

# Panel 2: Descenso de pérdida
ax2.plot(losses, 'teal', linewidth=2)
ax2.set_title("Curva de Convergencia del Backpropagation Matricial")
ax2.set_xlabel("Época")
ax2.set_ylabel("Cross-Entropy Loss")
ax2.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

---

## 4. ⚡ Transición a PyTorch Moderno

Constatemos que nuestro diseño modular en NumPy mapea 1 a 1 con la arquitectura `torch.nn.Module` de PyTorch:

In [ ]:
class PyTorchMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 2)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

t_X = torch.from_numpy(X_spiral.astype(np.float32))
t_y = torch.from_numpy(y_spiral).long()

torch_model = PyTorchMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(torch_model.parameters(), lr=0.2)

for epoch in range(800):
    optimizer.zero_grad()
    out = torch_model(t_X)
    loss = criterion(out, t_y)
    loss.backward()
    optimizer.step()

t_preds = torch.argmax(torch_model(t_X), dim=1).numpy()
print(f"Precisión PyTorch Oficial: {np.mean(t_preds == y_spiral) * 100:.1f}%")
print("✅ La estructura modular from-scratch coincide conceptualmente al 100% con PyTorch")

---

## 5. 🎯 Retos & Experimentos ("Tinker Time")

### Reto 1: Visualizar el Desvanecimiento del Gradiente (Sigmoide vs ReLU)
Vamos a demostrar empíricamente por qué la comunidad científica abandonó la sigmoide para redes profundas:
1. Construimos una red de 6 capas densas profundas con activación **Sigmoide**.
2. Construimos una red idéntica con activación **ReLU**.
3. Pasamos un lote hacia adelante, calculamos el backward y medimos la **norma del gradiente** en cada capa desde la salida (capa 6) hasta la primera capa oculta (capa 1).

In [ ]:
def simular_flujo_gradientes(activacion: str, n_layers: int = 6, dim: int = 20) -> List[float]:
    np.random.seed(42)
    X_test = np.random.randn(32, dim)
    weights = [np.random.randn(dim, dim) * 0.1 for _ in range(n_layers)]
    
    # Forward
    acts = [X_test]
    for W in weights:
        z = acts[-1] @ W
        if activacion == 'sigmoid':
            a = 1.0 / (1.0 + np.exp(-np.clip(z, -15, 15)))
        else:  # relu
            a = np.maximum(0.0, z)
        acts.append(a)
        
    # Backward
    grad = np.random.randn(32, dim)  # Gradiente entrante en la última capa
    normas = []
    for i in reversed(range(n_layers)):
        a = acts[i+1]
        if activacion == 'sigmoid':
            d_act = a * (1.0 - a)
        else:
            d_act = (a > 0.0).astype(float)
            
        delta = grad * d_act
        dW = acts[i].T @ delta
        normas.append(float(np.linalg.norm(dW)))
        grad = delta @ weights[i].T
        
    return normas[::-1]  # Ordenar de capa 1 (entrada) a capa N (salida)

normas_sigmoide = simular_flujo_gradientes('sigmoid', n_layers=6)
normas_relu = simular_flujo_gradientes('relu', n_layers=6)

capas = [f"Capa {i+1}" for i in range(6)]
plt.figure(figsize=(9, 4.5))
plt.plot(capas, normas_sigmoide, 'r-o', linewidth=2, label='Activación Sigmoide (Desvanecimiento masivo)')
plt.plot(capas, normas_relu, 'g-s', linewidth=2, label='Activación ReLU (Flujo de gradiente saludable)')
plt.yscale('log')
plt.title('Magnitud del Gradiente por Capa: El Misterio Resuelto del Vanishing Gradient')
plt.xlabel('Profundidad de la Red (De Entrada a Salida)')
plt.ylabel('Norma Frobenius del Gradiente dW (Escala Log)')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend()
plt.show()

print(f"Ratio de gradiente Capa 6 vs Capa 1 en Sigmoide: {normas_sigmoide[-1] / (normas_sigmoide[0] + 1e-12):.1f}x veces menor!")

### Reto 2 (Para resolver): Implementar Gradient Checking Matricial
Construye una función de verificación numérica que valide la derivada matricial respecto a $W$:
$$\frac{\partial \mathcal{L}}{\partial W_{i, j}} \approx \frac{\mathcal{L}(W_{i, j} + \epsilon) - \mathcal{L}(W_{i, j} - \epsilon)}{2\epsilon}$$
Calcula la diferencia relativa $\frac{\|dW_{analitico} - dW_{numerico}\|_2}{\|dW_{analitico}\|_2 + \|dW_{numerico}\|_2}$ y verifica que sea menor que $10^{-7}$.

In [ ]:
# TU CÓDIGO DEL RETO 2 AQUÍ
def matrix_gradient_check(layer: LinearLayer, X: np.ndarray, y_true: np.ndarray, eps: float = 1e-5) -> float:
    """
    Compara layer.dW analítico con la matriz de diferencias finitas numéricas.
    """
    # Tu código aquí
    pass

---

## 6. 📚 Referencias Fundamentales & Lecturas Recomendadas

### 📄 Papers Seminales
1. **Glorot, X., & Bengio, Y. (2010):** *"Understanding the difficulty of training deep feedforward neural networks"*, AISTATS 2010. [PMLR Link](http://proceedings.mlr.press/v9/glorot10a.html)
   * *¿Qué leer?* El análisis del comportamiento de la sigmoide y la saturación de neuronas en capas profundas.
2. **He, K., Zhang, X., Ren, S., & Sun, J. (2015):** *"Delving Deep into Rectifiers: Surpassing Human-Level Performance on ImageNet"*, ICCV 2015. [arXiv:1502.01852](https://arxiv.org/abs/1502.01852)
   * *¿Qué leer?* Sección 2.2: La derivación matemática formal de la inicialización de pesos Kaiming/He para ReLU ($Var(W) = 2 / n_{in}$).
3. **Rumelhart, D. E., Hinton, G. E., & Williams, R. J. (1986):** *"Learning representations by back-propagating errors"*, Nature, 323(6088), 533-536.

### 🔗 Libros de Referencia
* **Ian Goodfellow, Yoshua Bengio & Aaron Courville:** *Deep Learning* (MIT Press, 2016) - [Capítulo 6: Deep Feedforward Networks](https://www.deeplearningbook.org/contents/mlp.html).